In [0]:
dbutils.widgets.text("environment_name", "")
dbutils.widgets.text("volume_path", "/Volumes/delta_lake_catalog/default/deltalake_volume")

In [0]:
env_name = dbutils.widgets.get("environment_name")
volume_path = dbutils.widgets.get("volume_path")
raw_landing_path = f"{volume_path}/raw/orders"

print(f"environment name is {env_name}")
print(f"reading raw csv from {raw_landing_path}")

In [0]:
from pyspark.sql.functions import current_timestamp

orders_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_landing_path)
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env_name}_bronze")

(
    orders_df
        .withColumn("ingestion_time", current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{env_name}_bronze.orders")
)

In [0]:
display(spark.sql(f"SELECT * FROM {env_name}_bronze.orders LIMIT 20"))